In [1]:
pip install networkx matplotlib numpy pandas scikit-learn python-louvain python-igraph leidenalg torch

  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached python-louvain-0.16.tar.gz (204 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: ('Connection broken: IncompleteRead(215142 bytes read, 110872 more expected)', IncompleteRead(215142 bytes read, 110872 more expected))



In [2]:
pip install torch-geometric

  Using cached torch_geometric-2.8.0.post1-py3-none-any.whl.metadata (64 kB)
  Using cached aiohttp-3.14.3-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.0.1-cp314-cp314-win_amd64.whl.metadata (18 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
Using cached torch_geometric-2.8.0.post1-py3-none-any.whl (1.3 MB)
Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.7 MB 3.5 MB/s eta 0:00

In [4]:
D:\GNN Mini project\twitter_combined.txt

SyntaxError: unexpected character after line continuation character (3853362625.py, line 1)

In [ ]:
# ============================================================
# FINAL GNN PROJECT
# Community Detection + GNN Representation + Suspicious User Detection
#
# Dataset:
# twitter_combined.txt
#
# Algorithms:
# 1. Louvain
# 2. Spectral Clustering
# 3. Leiden
# 4. GNN Autoencoder
#
# Final output:
# - Community visualizations
# - GNN embedding visualization
# - Connectivity analysis
# - Suspicious user detection
# - Comparison CSV
# ============================================================

import os
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.cluster import SpectralClustering, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import community.community_louvain as community_louvain

import igraph as ig
import leidenalg

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = r"D:\GNN Mini project\twitter_combined.txt"

OUTPUT_DIR = r"D:\GNN Mini project\FINAL_RESULTS"

MAX_NODES = 5000

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# HELPER FUNCTION
# ============================================================

def save_figure(filename, dpi=300):

    path = os.path.join(OUTPUT_DIR, filename)

    plt.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight"
    )

    print("Saved:", path)


# ============================================================
# STEP 1 — LOAD TWITTER GRAPH
# ============================================================

print("\n" + "=" * 70)
print("STEP 1 — LOADING TWITTER DATASET")
print("=" * 70)

if not os.path.exists(DATASET_PATH):

    raise FileNotFoundError(
        f"\nDataset not found:\n{DATASET_PATH}\n\n"
        "Check the dataset path."
    )


G_full = nx.read_edgelist(
    DATASET_PATH,
    nodetype=int
)

print("Full Twitter Graph")
print("Nodes :", G_full.number_of_nodes())
print("Edges :", G_full.number_of_edges())


# ============================================================
# STEP 2 — CREATE CONNECTED 5000-NODE GRAPH
# ============================================================

print("\n" + "=" * 70)
print("STEP 2 — CREATING CONNECTED GRAPH")
print("=" * 70)

largest_component = max(
    nx.connected_components(G_full),
    key=len
)

G_connected = G_full.subgraph(largest_component).copy()

print("Largest connected component")
print("Nodes :", G_connected.number_of_nodes())
print("Edges :", G_connected.number_of_edges())


# Select 5000 nodes using BFS from a high-degree node.
# This preserves local connectivity better than random sampling.

if G_connected.number_of_nodes() > MAX_NODES:

    start_node = max(
        G_connected.degree,
        key=lambda x: x[1]
    )[0]

    selected_nodes = list(
        nx.bfs_tree(
            G_connected,
            start_node,
            depth_limit=None
        ).nodes()
    )[:MAX_NODES]

    G = G_connected.subgraph(selected_nodes).copy()

else:

    G = G_connected.copy()


# Ensure connectedness.
if not nx.is_connected(G):

    largest = max(
        nx.connected_components(G),
        key=len
    )

    G = G.subgraph(largest).copy()


print("\nFINAL ANALYSIS GRAPH")
print("Nodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())
print("Connected :", nx.is_connected(G))
print(
    "Average Degree :",
    round(
        sum(dict(G.degree()).values()) / G.number_of_nodes(),
        2
    )
)


# ============================================================
# NODE POSITIONS
# ============================================================

print("\nGenerating graph layout...")

pos = nx.spring_layout(
    G,
    seed=RANDOM_SEED,
    k=0.15,
    iterations=100
)


# ============================================================
# FIGURE 1 — ORIGINAL TWITTER NETWORK
# ============================================================

print("\nCreating original network visualization...")

plt.figure(figsize=(14, 10))

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.08,
    width=0.4
)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=8,
    alpha=0.75
)

plt.title(
    "Twitter Social Network — Node and Edge Connectivity",
    fontsize=18,
    fontweight="bold"
)

plt.axis("off")

save_figure("01_original_twitter_network.png")

plt.show()


# ============================================================
# STEP 3 — LOUVAIN
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — LOUVAIN COMMUNITY DETECTION")
print("=" * 70)

start = time.perf_counter()

louvain_partition = community_louvain.best_partition(
    G,
    random_state=RANDOM_SEED
)

louvain_time = time.perf_counter() - start

louvain_communities = {}

for node, community_id in louvain_partition.items():

    louvain_communities.setdefault(
        community_id,
        []
    ).append(node)


louvain_count = len(louvain_communities)

louvain_modularity = community_louvain.modularity(
    louvain_partition,
    G
)

louvain_sizes = [
    len(nodes)
    for nodes in louvain_communities.values()
]

print("Louvain Communities :", louvain_count)
print(
    "Modularity :",
    round(louvain_modularity, 4)
)

print(
    "Largest Community :",
    max(louvain_sizes)
)

print(
    "Average Community Size :",
    round(np.mean(louvain_sizes), 2)
)

print(
    "Execution Time :",
    round(louvain_time, 4),
    "seconds"
)


# ============================================================
# FIGURE 2 — LOUVAIN
# ============================================================

plt.figure(figsize=(14, 10))

louvain_colors = [
    louvain_partition[node]
    for node in G.nodes()
]

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.07,
    width=0.4
)

nodes_plot = nx.draw_networkx_nodes(
    G,
    pos,
    node_color=louvain_colors,
    cmap=plt.cm.tab20,
    node_size=10,
    alpha=0.85
)

plt.colorbar(
    nodes_plot,
    label="Louvain Community ID"
)

plt.title(
    "Louvain Community Detection on Twitter Network",
    fontsize=18,
    fontweight="bold"
)

plt.axis("off")

save_figure("02_louvain_communities.png")

plt.show()


# ============================================================
# STEP 4 — SPECTRAL CLUSTERING
# ============================================================

print("\n" + "=" * 70)
print("STEP 4 — SPECTRAL COMMUNITY DETECTION")
print("=" * 70)

nodes = list(G.nodes())

# Use Louvain community count as the number of clusters.
# This keeps the comparison based on the same graph.

spectral_k = max(
    2,
    min(louvain_count, 10)
)

print(
    "Number of Spectral Clusters :",
    spectral_k
)

start = time.perf_counter()

A = nx.to_numpy_array(
    G,
    nodelist=nodes
)

spectral_model = SpectralClustering(
    n_clusters=spectral_k,
    affinity="precomputed",
    assign_labels="kmeans",
    random_state=RANDOM_SEED
)

spectral_labels = spectral_model.fit_predict(A)

spectral_time = time.perf_counter() - start

spectral_partition = dict(
    zip(
        nodes,
        spectral_labels
    )
)

spectral_modularity = community_louvain.modularity(
    spectral_partition,
    G
)

spectral_sizes = np.bincount(
    spectral_labels
)

print(
    "Spectral Communities :",
    len(np.unique(spectral_labels))
)

print(
    "Modularity :",
    round(spectral_modularity, 4)
)

print(
    "Largest Community :",
    int(max(spectral_sizes))
)

print(
    "Average Community Size :",
    round(np.mean(spectral_sizes), 2)
)

print(
    "Execution Time :",
    round(spectral_time, 4),
    "seconds"
)


# ============================================================
# FIGURE 3 — SPECTRAL
# ============================================================

plt.figure(figsize=(14, 10))

spectral_colors = [
    spectral_partition[node]
    for node in G.nodes()
]

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.07,
    width=0.4
)

nodes_plot = nx.draw_networkx_nodes(
    G,
    pos,
    node_color=spectral_colors,
    cmap=plt.cm.tab20,
    node_size=10,
    alpha=0.85
)

plt.colorbar(
    nodes_plot,
    label="Spectral Cluster ID"
)

plt.title(
    "Spectral Community Detection on Twitter Network",
    fontsize=18,
    fontweight="bold"
)

plt.axis("off")

save_figure("03_spectral_communities.png")

plt.show()


# ============================================================
# STEP 5 — LEIDEN
# ============================================================

print("\n" + "=" * 70)
print("STEP 5 — LEIDEN COMMUNITY DETECTION")
print("=" * 70)

# Convert NetworkX graph to igraph.

igraph_graph = ig.Graph.from_networkx(G)

start = time.perf_counter()

leiden_result = leidenalg.find_partition(
    igraph_graph,
    leidenalg.ModularityVertexPartition,
    seed=RANDOM_SEED
)

leiden_time = time.perf_counter() - start

leiden_membership = leiden_result.membership

# NetworkX node ordering from Graph.from_networkx
leiden_partition = dict(
    zip(
        nodes,
        leiden_membership
    )
)

leiden_count = len(
    set(leiden_membership)
)

leiden_modularity = leiden_result.modularity

leiden_sizes = [
    len(group)
    for group in leiden_result
]

print(
    "Leiden Communities :",
    leiden_count
)

print(
    "Modularity :",
    round(leiden_modularity, 4)
)

print(
    "Largest Community :",
    max(leiden_sizes)
)

print(
    "Average Community Size :",
    round(np.mean(leiden_sizes), 2)
)

print(
    "Execution Time :",
    round(leiden_time, 4),
    "seconds"
)


# ============================================================
# FIGURE 4 — LEIDEN
# ============================================================

plt.figure(figsize=(14, 10))

leiden_colors = [
    leiden_partition[node]
    for node in G.nodes()
]

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.07,
    width=0.4
)

nodes_plot = nx.draw_networkx_nodes(
    G,
    pos,
    node_color=leiden_colors,
    cmap=plt.cm.tab20,
    node_size=10,
    alpha=0.85
)

plt.colorbar(
    nodes_plot,
    label="Leiden Community ID"
)

plt.title(
    "Leiden Community Detection on Twitter Network",
    fontsize=18,
    fontweight="bold"
)

plt.axis("off")

save_figure("04_leiden_communities.png")

plt.show()


# ============================================================
# STEP 6 — GRAPH CONNECTIVITY FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 — GRAPH CONNECTIVITY FEATURES")
print("=" * 70)

print("Calculating node features...")

degree_dict = dict(G.degree())

pagerank = nx.pagerank(
    G,
    max_iter=100
)

clustering = nx.clustering(G)

core_number = nx.core_number(G)

# Betweenness can be expensive.
# Sampling gives a practical approximation.

betweenness = nx.betweenness_centrality(
    G,
    k=min(500, G.number_of_nodes()),
    seed=RANDOM_SEED
)


# ============================================================
# STEP 7 — COMMUNITY FEATURES
# ============================================================

community_size = {}

for node, community_id in louvain_partition.items():

    community_size[node] = louvain_sizes[
        list(louvain_communities.keys()).index(
            community_id
        )
    ]


# ============================================================
# CREATE FEATURE DATAFRAME
# ============================================================

features = pd.DataFrame({
    "node": nodes,

    "degree": [
        degree_dict[n]
        for n in nodes
    ],

    "pagerank": [
        pagerank[n]
        for n in nodes
    ],

    "clustering": [
        clustering[n]
        for n in nodes
    ],

    "core_number": [
        core_number[n]
        for n in nodes
    ],

    "betweenness": [
        betweenness[n]
        for n in nodes
    ],

    "louvain_community": [
        louvain_partition[n]
        for n in nodes
    ],

    "spectral_cluster": [
        spectral_partition[n]
        for n in nodes
    ],

    "leiden_community": [
        leiden_partition[n]
        for n in nodes
    ]
})


# ============================================================
# STEP 8 — GRAPH-BASED SUSPICIOUS USER SCORE
# ============================================================

print("\n" + "=" * 70)
print("STEP 8 — SUSPICIOUS USER / SPAM-RISK DETECTION")
print("=" * 70)

# Standardize structural features.

risk_features = [
    "degree",
    "pagerank",
    "clustering",
    "core_number",
    "betweenness"
]

X_risk = features[risk_features].values

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_risk)


# ------------------------------------------------------------
# Structural anomaly score
# ------------------------------------------------------------

# High absolute deviation from the average structural profile
# indicates an unusual structural pattern.

anomaly_score = np.mean(
    np.abs(X_scaled),
    axis=1
)

# Normalize between 0 and 1.

anomaly_min = anomaly_score.min()
anomaly_max = anomaly_score.max()

if anomaly_max > anomaly_min:

    risk_score = (
        anomaly_score - anomaly_min
    ) / (
        anomaly_max - anomaly_min
    )

else:

    risk_score = np.zeros_like(
        anomaly_score
    )


features["risk_score"] = risk_score


# ------------------------------------------------------------
# Risk categories
# ------------------------------------------------------------

features["status"] = np.where(
    features["risk_score"] >= 0.80,
    "HIGH RISK",
    np.where(
        features["risk_score"] >= 0.60,
        "MEDIUM RISK",
        "NORMAL"
    )
)


high_risk_count = np.sum(
    features["status"] == "HIGH RISK"
)

medium_risk_count = np.sum(
    features["status"] == "MEDIUM RISK"
)

normal_count = np.sum(
    features["status"] == "NORMAL"
)


print("High-risk users   :", high_risk_count)
print("Medium-risk users :", medium_risk_count)
print("Normal users      :", normal_count)


# ============================================================
# FIGURE 5 — RISK VISUALIZATION
# ============================================================

plt.figure(figsize=(14, 10))

# Normal nodes
normal_mask = (
    features["status"] == "NORMAL"
)

medium_mask = (
    features["status"] == "MEDIUM RISK"
)

high_mask = (
    features["status"] == "HIGH RISK"
)

normal_nodes = features.loc[
    normal_mask,
    "node"
].tolist()

medium_nodes = features.loc[
    medium_mask,
    "node"
].tolist()

high_nodes = features.loc[
    high_mask,
    "node"
].tolist()


# Edges

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.05,
    width=0.4
)


# Normal

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=normal_nodes,
    node_size=8,
    alpha=0.35,
    label="Normal"
)


# Medium risk

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=medium_nodes,
    node_size=15,
    alpha=0.7,
    label="Medium Risk"
)


# High risk

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=high_nodes,
    node_size=30,
    alpha=0.95,
    label="High Risk / Suspicious"
)


plt.title(
    "Community-Aware Suspicious User Detection",
    fontsize=18,
    fontweight="bold"
)

plt.legend()

plt.axis("off")

save_figure("05_suspicious_user_detection.png")

plt.show()


# ============================================================
# STEP 9 — GNN GRAPH REPRESENTATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 9 — GNN GRAPH REPRESENTATION")
print("=" * 70)


# ------------------------------------------------------------
# Node features
# ------------------------------------------------------------

gnn_features = features[
    [
        "degree",
        "pagerank",
        "clustering",
        "core_number",
        "betweenness"
    ]
].values.astype(np.float32)


gnn_scaler = StandardScaler()

gnn_features = gnn_scaler.fit_transform(
    gnn_features
).astype(np.float32)


x = torch.tensor(
    gnn_features,
    dtype=torch.float
)


# ------------------------------------------------------------
# Edge index
# ------------------------------------------------------------

node_to_index = {
    node: i
    for i, node in enumerate(nodes)
}

edge_list = []

for u, v in G.edges():

    edge_list.append([
        node_to_index[u],
        node_to_index[v]
    ])

    edge_list.append([
        node_to_index[v],
        node_to_index[u]
    ])


edge_index = torch.tensor(
    edge_list,
    dtype=torch.long
).t().contiguous()


# ============================================================
# GCN AUTOENCODER
# ============================================================

class GCNEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        embedding_dim=16
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            embedding_dim
        )


    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x


# ============================================================
# TRAIN GNN
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "GNN Device :",
    device
)


x = x.to(device)
edge_index = edge_index.to(device)


model = GCNEncoder(
    input_dim=x.shape[1],
    hidden_dim=32,
    embedding_dim=16
).to(device)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)


# ============================================================
# TRAINING
# ============================================================

model.train()

loss_history = []


for epoch in range(1, 101):

    optimizer.zero_grad()

    z = model(
        x,
        edge_index
    )

    # Link reconstruction objective

    src = edge_index[0]
    dst = edge_index[1]

    positive_scores = (
        z[src] * z[dst]
    ).sum(dim=1)

    positive_loss = -torch.log(
        torch.sigmoid(
            positive_scores
        ) + 1e-8
    ).mean()


    # Negative samples

    negative_dst = torch.randint(
        0,
        x.shape[0],
        (
            edge_index.shape[1],
        ),
        device=device
    )

    negative_scores = (
        z[src] * z[negative_dst]
    ).sum(dim=1)

    negative_loss = -torch.log(
        1 -
        torch.sigmoid(
            negative_scores
        ) + 1e-8
    ).mean()


    loss = (
        positive_loss +
        negative_loss
    )

    loss.backward()

    optimizer.step()

    loss_history.append(
        loss.item()
    )


    if epoch % 20 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f}"
        )


# ============================================================
# GET EMBEDDINGS
# ============================================================

model.eval()

with torch.no_grad():

    embeddings = model(
        x,
        edge_index
    ).cpu().numpy()


np.save(
    os.path.join(
        OUTPUT_DIR,
        "gnn_node_embeddings.npy"
    ),
    embeddings
)


# ============================================================
# FIGURE 6 — GNN TRAINING LOSS
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    loss_history,
    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Reconstruction Loss")

plt.title(
    "GNN Representation Learning",
    fontsize=18,
    fontweight="bold"
)

plt.grid(alpha=0.25)

save_figure(
    "06_gnn_training_loss.png"
)

plt.show()


# ============================================================
# STEP 10 — GNN EMBEDDING VISUALIZATION
# ============================================================

print("\nCreating GNN embedding visualization...")

tsne = TSNE(
    n_components=2,
    random_state=RANDOM_SEED,
    perplexity=30,
    init="pca",
    learning_rate="auto"
)

embedding_2d = tsne.fit_transform(
    embeddings
)


plt.figure(figsize=(14, 10))

scatter = plt.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1],
    c=features["louvain_community"],
    cmap="tab20",
    s=10,
    alpha=0.75
)

plt.colorbar(
    scatter,
    label="Louvain Community"
)

plt.title(
    "GNN Learned Node Representations",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("GNN Embedding Dimension 1")
plt.ylabel("GNN Embedding Dimension 2")

save_figure(
    "07_gnn_embeddings.png"
)

plt.show()


# ============================================================
# STEP 11 — COMBINE GNN REPRESENTATION WITH RISK FEATURES
# ============================================================

print("\nCombining GNN embeddings with graph features...")


embedding_norm = np.linalg.norm(
    embeddings,
    axis=1
)


embedding_z = (
    embedding_norm -
    embedding_norm.mean()
) / (
    embedding_norm.std() + 1e-8
)


# Combine structural anomaly and GNN embedding magnitude.

combined_score = (
    0.70 * risk_score +
    0.30 * (
        1 /
        (
            1 +
            np.exp(-embedding_z)
        )
    )
)


# Normalize.

combined_min = combined_score.min()
combined_max = combined_score.max()

final_risk = (
    combined_score - combined_min
) / (
    combined_max - combined_min + 1e-8
)


features["gnn_embedding_norm"] = embedding_norm

features["final_risk_score"] = final_risk


features["final_status"] = np.where(
    final_risk >= 0.80,
    "HIGH RISK / SUSPICIOUS",
    np.where(
        final_risk >= 0.60,
        "MEDIUM RISK",
        "NORMAL"
    )
)


# ============================================================
# FIGURE 7 — FINAL GNN + SPAM RISK
# ============================================================

print("\nCreating final GNN + suspicious-user visualization...")


plt.figure(figsize=(14, 10))

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.04,
    width=0.4
)


final_normal = features[
    features["final_status"] == "NORMAL"
]["node"].tolist()


final_medium = features[
    features["final_status"] == "MEDIUM RISK"
]["node"].tolist()


final_high = features[
    features["final_status"] ==
    "HIGH RISK / SUSPICIOUS"
]["node"].tolist()


nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=final_normal,
    node_size=7,
    alpha=0.25,
    label="Normal"
)


nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=final_medium,
    node_size=16,
    alpha=0.65,
    label="Medium Risk"
)


nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=final_high,
    node_size=35,
    alpha=0.95,
    label="High Risk / Suspicious"
)


plt.title(
    "Final Community-Aware GNN Suspicious User Detection",
    fontsize=18,
    fontweight="bold"
)

plt.legend()

plt.axis("off")

save_figure(
    "08_final_gnn_spam_risk_network.png"
)

plt.show()


# ============================================================
# STEP 12 — TOP SUSPICIOUS USERS
# ============================================================

print("\n" + "=" * 70)
print("TOP SUSPICIOUS USERS")
print("=" * 70)


top_suspicious = features.sort_values(
    "final_risk_score",
    ascending=False
).head(20)


print(
    top_suspicious[
        [
            "node",
            "degree",
            "pagerank",
            "louvain_community",
            "spectral_cluster",
            "leiden_community",
            "final_risk_score",
            "final_status"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# STEP 13 — FINAL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame({

    "Algorithm": [
        "Louvain",
        "Spectral",
        "Leiden"
    ],

    "Nodes": [
        G.number_of_nodes(),
        G.number_of_nodes(),
        G.number_of_nodes()
    ],

    "Edges": [
        G.number_of_edges(),
        G.number_of_edges(),
        G.number_of_edges()
    ],

    "Communities": [
        louvain_count,
        len(np.unique(spectral_labels)),
        leiden_count
    ],

    "Modularity": [
        louvain_modularity,
        spectral_modularity,
        leiden_modularity
    ],

    "Execution_Time_sec": [
        louvain_time,
        spectral_time,
        leiden_time
    ],

    "Average_Community_Size": [
        np.mean(louvain_sizes),
        np.mean(spectral_sizes),
        np.mean(leiden_sizes)
    ],

    "Largest_Community": [
        max(louvain_sizes),
        max(spectral_sizes),
        max(leiden_sizes)
    ]
})


comparison = comparison.round(4)


comparison_path = os.path.join(
    OUTPUT_DIR,
    "algorithm_comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)


print("\n" + "=" * 70)
print("ALGORITHM COMPARISON")
print("=" * 70)

print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# STEP 14 — SAVE NODE RESULTS
# ============================================================

node_results_path = os.path.join(
    OUTPUT_DIR,
    "node_connectivity_and_spam_risk.csv"
)

features.to_csv(
    node_results_path,
    index=False
)


# ============================================================
# STEP 15 — SAVE TOP SUSPICIOUS USERS
# ============================================================

top_path = os.path.join(
    OUTPUT_DIR,
    "top_suspicious_users.csv"
)

top_suspicious.to_csv(
    top_path,
    index=False
)


# ============================================================
# STEP 16 — COMMUNITY SIZE VISUALIZATION
# ============================================================

plt.figure(figsize=(12, 7))

methods = [
    "Louvain",
    "Spectral",
    "Leiden"
]

largest_values = [
    max(louvain_sizes),
    max(spectral_sizes),
    max(leiden_sizes)
]

plt.bar(
    methods,
    largest_values
)

plt.ylabel(
    "Number of Nodes"
)

plt.title(
    "Largest Community Size Comparison",
    fontsize=18,
    fontweight="bold"
)

plt.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "09_largest_community_comparison.png"
)

plt.show()


# ============================================================
# STEP 17 — MODULARITY COMPARISON
# ============================================================

plt.figure(figsize=(10, 6))

modularity_values = [
    louvain_modularity,
    spectral_modularity,
    leiden_modularity
]

plt.bar(
    methods,
    modularity_values
)

plt.ylabel(
    "Modularity"
)

plt.title(
    "Community Detection Modularity Comparison",
    fontsize=18,
    fontweight="bold"
)

plt.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "10_modularity_comparison.png"
)

plt.show()


# ============================================================
# STEP 18 — RISK DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

risk_counts = [
    np.sum(
        features["final_status"] ==
        "NORMAL"
    ),

    np.sum(
        features["final_status"] ==
        "MEDIUM RISK"
    ),

    np.sum(
        features["final_status"] ==
        "HIGH RISK / SUSPICIOUS"
    )
]

plt.bar(
    [
        "Normal",
        "Medium Risk",
        "High Risk"
    ],
    risk_counts
)

plt.ylabel(
    "Number of Users"
)

plt.title(
    "Suspicious User Risk Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.grid(
    axis="y",
    alpha=0.25
)

save_figure(
    "11_risk_distribution.png"
)

plt.show()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL PROJECT RESULTS")
print("=" * 70)

print(
    f"Twitter nodes analysed      : {G.number_of_nodes()}"
)

print(
    f"Twitter edges analysed      : {G.number_of_edges()}"
)

print(
    f"Louvain communities         : {louvain_count}"
)

print(
    f"Spectral communities        : {len(np.unique(spectral_labels))}"
)

print(
    f"Leiden communities          : {leiden_count}"
)

print(
    f"GNN embedding dimension    : {embeddings.shape[1]}"
)

print(
    f"High-risk users detected    : "
    f"{np.sum(features['final_status'] == 'HIGH RISK / SUSPICIOUS')}"
)

print(
    f"Medium-risk users detected  : "
    f"{np.sum(features['final_status'] == 'MEDIUM RISK')}"
)

print(
    f"Normal users                : "
    f"{np.sum(features['final_status'] == 'NORMAL')}"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        " -",
        filename
    )

print("\n" + "=" * 70)
print("PROJECT EXECUTION COMPLETED")
print("=" * 70)

ModuleNotFoundError: No module named 'pandas'

In [11]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'